In [1]:
import pickle

import kagglehub
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

/home/yesavage/Documents/ai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Download dataset
path = kagglehub.dataset_download("fedesoriano/company-bankruptcy-prediction")
print("Path to dataset files:", path)

Path to dataset files: /home/yesavage/.cache/kagglehub/datasets/fedesoriano/company-bankruptcy-prediction/versions/2


In [3]:
# Load data
train_data = pd.read_csv(
    "/home/yesavage/.cache/kagglehub/datasets/fedesoriano/company-bankruptcy-prediction/versions/2/data.csv"
)


def clean_col(col):
    return (
        col.strip()  # removes hidden leading spaces
        .replace(" ", "_")
        .replace("%", "percent")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("¥", "Yuan")
    )


train_data.columns = [clean_col(c) for c in train_data.columns]

print(train_data.columns)
print(train_data.shape)

Index(['Bankrupt?', 'ROAC_before_interest_and_depreciation_before_interest',
       'ROAA_before_interest_and_percent_after_tax',
       'ROAB_before_interest_and_depreciation_after_tax',
       'Operating_Gross_Margin', 'Realized_Sales_Gross_Margin',
       'Operating_Profit_Rate', 'Pre-tax_net_Interest_Rate',
       'After-tax_net_Interest_Rate',
       'Non-industry_income_and_expenditure_revenue',
       'Continuous_interest_rate_after_tax', 'Operating_Expense_Rate',
       'Research_and_development_expense_rate', 'Cash_flow_rate',
       'Interest-bearing_debt_interest_rate', 'Tax_rate_A',
       'Net_Value_Per_Share_B', 'Net_Value_Per_Share_A',
       'Net_Value_Per_Share_C', 'Persistent_EPS_in_the_Last_Four_Seasons',
       'Cash_Flow_Per_Share', 'Revenue_Per_Share_Yuan_Yuan',
       'Operating_Profit_Per_Share_Yuan_Yuan',
       'Per_Share_Net_profit_before_tax_Yuan_Yuan',
       'Realized_Sales_Gross_Profit_Growth_Rate',
       'Operating_Profit_Growth_Rate', 'After-tax_Net_Pr

In [4]:
# Split features and target
X = train_data.drop("Bankrupt?", axis=1)
y = train_data["Bankrupt?"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
# Scale full dataset for RFE
scaler_rfe = StandardScaler()
X_train_scaled_full = scaler_rfe.fit_transform(X_train)

In [6]:
# Feature selection using RFE
model = LogisticRegression(max_iter=500, class_weight="balanced")
rfe = RFE(model, n_features_to_select=10)
rfe.fit(X_train_scaled_full, y_train)

selected_mask = rfe.support_
selected_features = X.columns[selected_mask]

print("Selected features:")
print(selected_features)

Selected features:
Index(['Net_Value_Per_Share_B', 'Net_Value_Per_Share_C',
       'Persistent_EPS_in_the_Last_Four_Seasons',
       'Operating_Profit_Per_Share_Yuan_Yuan', 'Debt_ratio_percent',
       'Net_worth_Assets', 'Borrowing_dependency',
       'Current_Liability_to_Assets', 'Current_Liability_to_Liability',
       'Liability_to_Equity'],
      dtype='str')


In [7]:
# Select only chosen features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

In [8]:
# Final scaling
final_scaler = StandardScaler()
X_train_scaled = final_scaler.fit_transform(X_train_selected)
X_test_scaled = final_scaler.transform(X_test_selected)

In [9]:
# Train final model
final_model = LogisticRegression(max_iter=2000, class_weight="balanced")
final_model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [10]:
# Predictions
y_pred = final_model.predict(X_test_scaled)
y_prob = final_model.predict_proba(X_test_scaled)[:, 1]

In [11]:
# Evaluation
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("Log Loss:", log_loss(y_test, y_prob))

F1 Score: 0.3111111111111111
ROC-AUC: 0.9259143108881025
Log Loss: 0.34088409908929423


In [12]:
pickle.dump(final_model, open("model.pkl", "wb"))
pickle.dump(final_scaler, open("scaler.pkl", "wb"))
pickle.dump(selected_features.tolist(), open("selected_features.pkl", "wb"))

print("Model, scaler, and selected features saved successfully!")

Model, scaler, and selected features saved successfully!
